# Mission 3 전처리 — 규정 준수 텍스트 정제 및 9개 타겟 증상 CSV 생성

원본 라벨 JSON에서 대화 본문(`utterances[].text`)만 추출해 학습용 CSV를 만든다.
전처리 규칙은 `m3.labels`를 그대로 재사용한다. 이 노트북에 로직을 복제하지 않는다.
복제하면 한쪽만 수정됐을 때 학습 CSV와 추론 경로가 조용히 어긋난다.

발화 경계(턴 구분) 표현은 `UTTERANCE_SEP_MODE`로 선택한다. 대회 Q&A 답변(2026-09-20)으로
`speaker` 값을 사용하지 않고 발화 경계만 남기는 전처리는 허용됐다.


## 0-A. 저장소 준비 (Colab 전용)

이 노트북은 전처리 로직을 복제하지 않고 저장소의 `m3` 모듈을 호출한다. 따라서 저장소가 먼저 있어야 한다.
로컬처럼 이미 저장소 안에서 실행 중이면 이 셀은 아무것도 하지 않는다.

In [ ]:
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/KozzilzzilE/DCC_Problem.git"
BRANCH = "mission3-utterance-boundary"
REPO_DIR = Path("/content/DCC_Problem")

already_in_repo = any((Path.cwd() / c / "m3").is_dir() for c in (".", "mission3_symptom"))
if already_in_repo:
    print("저장소 안에서 실행 중입니다. clone 을 건너뜁니다.")
elif REPO_DIR.is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", BRANCH], check=True)
    print(f"기존 clone 갱신: {REPO_DIR} ({BRANCH})")
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    print(f"clone 완료: {REPO_DIR} ({BRANCH})")

## 0. 환경 및 모듈 준비

In [ ]:
import json
import re
import sys
from pathlib import Path

import pandas as pd
from tqdm import tqdm

# 저장소의 mission3_symptom 디렉터리를 찾아 m3 모듈을 불러온다 (model_train.ipynb과 동일한 방식).
CANDIDATES = [
    Path.cwd(),
    Path.cwd() / "mission3_symptom",
    Path("/content/DCC_Problem/mission3_symptom"),
    Path("/content/drive/MyDrive/DCC_Problem/mission3_symptom"),
]
mission3_dir = next((p for p in CANDIDATES if (p / "m3").is_dir()), None)
if mission3_dir is None:
    raise RuntimeError(
        "mission3_symptom 디렉터리를 찾지 못했습니다. 저장소를 clone 한 뒤 "
        "repository root 또는 mission3_symptom 에서 실행하세요."
    )
sys.path.insert(0, str(mission3_dir))

from m3.config import TARGET_SYMPTOMS, UTTERANCE_SEP_MODES
from m3.labels import load_transcripts_dir

print(f"Mission 3 directory: {mission3_dir.resolve()}")
print(f"사용 가능한 발화 경계 모드: {sorted(UTTERANCE_SEP_MODES)}")

## 1. 설정

`UTTERANCE_SEP_MODE`만 바꾸면 같은 원본에서 다른 입력 표현의 CSV가 만들어진다.

| 모드 | 결합 방식 | 비고 |
|---|---|---|
| `space` | `"발화1 발화2"` | 기존 baseline. 경계 정보 없음 |
| `sep` | `"발화1 [SEP] 발화2"` | tokenizer 기본 어휘의 경계 토큰 재사용. vocab 변경 불필요 |
| `turn` | `"발화1 [TURN] 발화2"` | 전용 special token. tokenizer 등록과 embedding resize 선행 필요 |

개행(`\n`)은 모드로 제공하지 않는다. BERT 계열 tokenizer는 basic tokenization에서 개행을
공백과 동일하게 처리하므로 token 열이 바뀌지 않아 `space`와 결과가 같다.

In [ ]:
UTTERANCE_SEP_MODE = "sep"   # "space" | "sep" | "turn"

# 원본 라벨 zip 이 있는 Drive 경로. 공유드라이브는 MyDrive 가 아니라 Shareddrives 아래에 마운트된다.
#   내 드라이브   : /content/drive/MyDrive/...
#   공유 드라이브 : /content/drive/Shareddrives/<드라이브 이름>/...
# None 이면 두 위치를 모두 탐색해 자동으로 찾는다.
DRIVE_ROOT = None          # 예: "/content/drive/Shareddrives/코찔찔이"
LABELS_ZIP_NAME = "mission3_labels.zip"

TRAIN_LABEL_DIR = Path("/content/train/label")
VAL_LABEL_DIR = Path("/content/val/label")

print(f"모드: {UTTERANCE_SEP_MODE!r} -> 구분자 {UTTERANCE_SEP_MODES[UTTERANCE_SEP_MODE]!r}")

## 2. 원본 라벨 압축 해제

이미 `/content`에 풀려 있으면 건너뛴다.

In [ ]:
import glob
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

SEARCH_ROOTS = [Path(DRIVE_ROOT)] if DRIVE_ROOT else [
    Path("/content/drive/MyDrive"),
    *sorted(Path("/content/drive/Shareddrives").glob("*")),
]
print("탐색 대상 Drive 경로:")
for root in SEARCH_ROOTS:
    print(f"  - {root}")

if not TRAIN_LABEL_DIR.is_dir():
    zip_path = None
    for root in SEARCH_ROOTS:
        hits = glob.glob(str(root / "**" / LABELS_ZIP_NAME), recursive=True)
        if hits:
            zip_path = Path(hits[0])
            break
    if zip_path is None:
        raise FileNotFoundError(
            f"{LABELS_ZIP_NAME} 을 찾지 못했습니다. DRIVE_ROOT 를 직접 지정하세요. "
            f"공유드라이브는 /content/drive/Shareddrives/<드라이브 이름> 입니다."
        )
    print(f"\n압축 해제: {zip_path}")
    !unzip -q "{zip_path}" -d /content/
    DRIVE_SOURCE = zip_path.parent
else:
    DRIVE_SOURCE = SEARCH_ROOTS[0]

# 압축 구조에 따라 /content/data/ 아래에 풀릴 수 있다.
if not TRAIN_LABEL_DIR.is_dir():
    TRAIN_LABEL_DIR = Path("/content/data/train/label")
    VAL_LABEL_DIR = Path("/content/data/val/label")

# 공유드라이브는 쓰기 권한이 없을 수 있으므로 저장 위치를 실제로 검사해서 고른다.
def is_writable(path: Path) -> bool:
    try:
        path.mkdir(parents=True, exist_ok=True)
        probe = path / ".write_test"
        probe.write_text("ok", encoding="utf-8")
        probe.unlink()
        return True
    except OSError:
        return False

OUTPUT_DIR = next(
    (p for p in (DRIVE_SOURCE, Path("/content/drive/MyDrive")) if is_writable(p)),
    Path("/content/output"),
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

suffix = "" if UTTERANCE_SEP_MODE == "space" else f"_{UTTERANCE_SEP_MODE}"
TRAIN_CSV = OUTPUT_DIR / f"mission3_train{suffix}.csv"
VAL_CSV = OUTPUT_DIR / f"mission3_val{suffix}.csv"
PROVENANCE_PATH = OUTPUT_DIR / f"mission3_preprocess{suffix}.json"

print(f"\ntrain label dir: {TRAIN_LABEL_DIR} ({len(list(TRAIN_LABEL_DIR.glob('*.json')))}건)")
print(f"val   label dir: {VAL_LABEL_DIR} ({len(list(VAL_LABEL_DIR.glob('*.json')))}건)")
print(f"출력 경로: {OUTPUT_DIR}")
if OUTPUT_DIR == Path("/content/output"):
    print("  주의: Drive 에 쓸 수 없어 세션 로컬에 저장합니다. 런타임 종료 시 사라지므로 직접 내려받으세요.")

## 3. 전처리 실행

`m3.labels.load_transcripts_dir`가 화자·시간·인적사항을 파싱 단계에서 원천 배제하고,
9개 타겟 외 증상을 라벨 벡터에서 제거한다. 학습이 요구하는 9개 이진 컬럼을 여기서 펼친다.

In [ ]:
def build_dataframe(label_dir: Path, sep_mode: str) -> pd.DataFrame:
    records = load_transcripts_dir(label_dir, sep_mode=sep_mode)
    if not records:
        raise RuntimeError(f"파싱된 레코드가 없습니다: {label_dir}")

    rows = []
    for record in tqdm(records, desc=str(label_dir)):
        row = {
            "call_id": record.call_id,
            "text": record.text,
            "symptoms": str(sorted(record.symptoms)),
            "label_vector": str(record.label_vector.tolist()),
        }
        for idx, symptom in enumerate(TARGET_SYMPTOMS):
            row[symptom] = int(record.label_vector[idx])
        rows.append(row)
    return pd.DataFrame(rows)


train_df = build_dataframe(TRAIN_LABEL_DIR, UTTERANCE_SEP_MODE)
val_df = build_dataframe(VAL_LABEL_DIR, UTTERANCE_SEP_MODE)

train_df.to_csv(TRAIN_CSV, index=False, encoding="utf-8-sig")
val_df.to_csv(VAL_CSV, index=False, encoding="utf-8-sig")
print(f"\nTrain {len(train_df):,}건 -> {TRAIN_CSV}")
print(f"Val   {len(val_df):,}건 -> {VAL_CSV}")

## 4. 발화 경계 도입 영향 측정 (재학습 전 필수)

구분자를 넣으면 입력 길이가 늘어 512 token 초과 비율이 올라간다. 그 상승폭을 먼저 재지 않으면
"경계 정보의 효과"와 "절단 증가의 부작용"이 섞여 실험 결과를 해석할 수 없다.

baseline(`space`)의 Validation 512 token 초과 비율은 KLUE-RoBERTa 기준 `98 / 3,640 = 2.69%`였다.

In [ ]:
space_val = build_dataframe(VAL_LABEL_DIR, "space")
merged = space_val[["call_id", "text"]].merge(
    val_df[["call_id", "text"]], on="call_id", suffixes=("_space", "_sep")
)

marker = UTTERANCE_SEP_MODES[UTTERANCE_SEP_MODE].strip()
if not marker:
    raise RuntimeError("space 모드에서는 경계 영향을 측정할 대상이 없습니다.")

turns = merged["text_sep"].str.count(re.escape(marker)) + 1
print(f"발화 수(턴) 분포: 평균 {turns.mean():.1f} / 중앙값 {turns.median():.0f} "
      f"/ p90 {turns.quantile(0.9):.0f} / 최대 {turns.max():.0f}")
print(f"문자 길이 증가: 평균 {(merged['text_sep'].str.len() - merged['text_space'].str.len()).mean():.0f}자")

실제 token 수는 tokenizer로 재야 한다. 아래 셀은 학습에 쓸 backbone과 동일한 tokenizer로
512 초과 비율이 얼마나 올라가는지 측정하고 결과를 provenance JSON에 남긴다.

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "klue/roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def overflow_rate(texts, max_length: int = 512) -> dict:
    lengths = [len(tokenizer(text, add_special_tokens=True)["input_ids"]) for text in tqdm(texts)]
    over = sum(1 for n in lengths if n > max_length)
    return {
        "count": len(lengths),
        "over_512": over,
        "over_512_rate": over / len(lengths),
        "mean_tokens": sum(lengths) / len(lengths),
        "max_tokens": max(lengths),
    }

space_stats = overflow_rate(merged["text_space"].tolist())
sep_stats = overflow_rate(merged["text_sep"].tolist())

print(f"space : 초과 {space_stats['over_512']}/{space_stats['count']} "
      f"({space_stats['over_512_rate']:.2%}), 평균 {space_stats['mean_tokens']:.0f} token")
print(f"{UTTERANCE_SEP_MODE:6s}: 초과 {sep_stats['over_512']}/{sep_stats['count']} "
      f"({sep_stats['over_512_rate']:.2%}), 평균 {sep_stats['mean_tokens']:.0f} token")
print(f"=> 초과 비율 변화: {sep_stats['over_512_rate'] - space_stats['over_512_rate']:+.2%}")

# 구분자가 실제로 단일 특수 토큰으로 처리되는지 확인. 여러 토큰으로 쪼개지면 길이 비용이 커진다.
probe = tokenizer(f"첫 발화{UTTERANCE_SEP_MODES[UTTERANCE_SEP_MODE]}둘째 발화", add_special_tokens=False)
print(f"구분자 토큰화 확인: {tokenizer.convert_ids_to_tokens(probe['input_ids'])}")

## 5. 전처리 이력 기록

보고서의 재현 절차와 규정 대응 근거로 쓴다.

In [ ]:
provenance = {
    "utterance_sep_mode": UTTERANCE_SEP_MODE,
    "separator": UTTERANCE_SEP_MODES[UTTERANCE_SEP_MODE],
    "model_for_token_stats": MODEL_NAME,
    "train_rows": int(len(train_df)),
    "val_rows": int(len(val_df)),
    "train_csv": TRAIN_CSV.name,
    "val_csv": VAL_CSV.name,
    "val_token_stats": {"space": space_stats, UTTERANCE_SEP_MODE: sep_stats},
    "label_source": "utterances[].text only (speaker/startAt/endAt/인적사항 원천 배제)",
    "rule_basis": "대회 Q&A 답변(2026-09-20): speaker 미사용, 발화 경계 구분만 허용",
}
PROVENANCE_PATH.write_text(json.dumps(provenance, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(provenance, ensure_ascii=False, indent=2))

print("\n[9개 타겟 증상별 데이터 분포]")
print(pd.DataFrame({
    "Train 건수": train_df[TARGET_SYMPTOMS].sum(),
    "Val 건수": val_df[TARGET_SYMPTOMS].sum(),
}))